# Training Table Walkthrough

Builds a training table from `pre_training_table.parquet`.

| Column | Description |
|---|---|
| `t.match_id` | Match identifier |
| `t.period` | Period 1 or 2 |
| `window_time` | Window end time in seconds (0.5, 1.0, 1.5 …) |
| `data_split` | `train` / `validation` / `test` |
| `p.event_label` | Primary event in the window (`PASS`, `TACKLE`, `no event`, …) |
| `is_pass` | Binary target: 1 = PASS, 0 = anything else |
| `ball_speed_avg_xy` | Average 2-D ball speed across the window (m / frame) |
| `primary_event_frame` | `t.frame` of the pre-training row used as primary event. `NaN` for `no event` windows. |
| `closest_player_id` | ID of the visible player closest to the ball at the primary event frame. `NaN` if no visible player or ball position missing. |
| `closest_player_team_id` | Team of the closest visible player at the primary event frame. `NaN` if no visible player or ball position missing. |

## 1. Load data

In [168]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import numpy as np
import pandas as pd
from IPython.display import display

from driblab.config import CONFIG_PATH, MODEL_BASE_DATA_DIR
from driblab.features import match_splits

pd.set_option('display.max_columns', None)

PRE_TRAINING_PATH = MODEL_BASE_DATA_DIR / 'pre_training_table.parquet'
OUTPUT_PATH       = MODEL_BASE_DATA_DIR / 'training_table_simple.parquet'

assert PRE_TRAINING_PATH.exists(), f'Missing {PRE_TRAINING_PATH} — run pre_training_table.ipynb first.'

df = pd.read_parquet(PRE_TRAINING_PATH)
print(f'Loaded {len(df):,} rows')
display(df[['t.match_id', 't.period', 't.frame', 't.Videotimestamp',
            'p.event_label', 'p.dist_to_actual_event',
            't.ball_x', 't.ball_y']].head(6))

Loaded 1,986,630 rows


,t.match_id,t.period,t.frame,t.Videotimestamp,p.event_label,p.dist_to_actual_event,t.ball_x,t.ball_y
0,678949,1,0,1.0,PASS,0.2,NaN,NaN
1,678949,1,1,1.1,PASS,0.1,NaN,NaN
2,678949,1,2,1.2,PASS,0.0,NaN,NaN
3,678949,1,3,1.3,PASS,0.1,NaN,NaN
4,678949,1,4,1.4,PASS,0.2,NaN,NaN
5,678949,1,5,1.5,PASS,0.3,NaN,NaN


## 2. Add data_split column

Split assignments come from `config.yaml` and are applied at the **match level** — every frame from the same match lands in the same split.
This prevents the model from seeing frames from match X in training and other frames from the same match in the test set (data leakage).

In [169]:
splits = match_splits.load_match_splits(CONFIG_PATH)

for split_name, ids in splits.items():
    print(f'  {split_name:12s}: {len(ids)} matches')

df = match_splits.add_data_split_column(df, splits, match_col='t.match_id')

print()
print('Frames per split:')
display(
    df.groupby('data_split').agg(
        frames=('t.match_id', 'count'),
        matches=('t.match_id', 'nunique'),
    ).reset_index()
)

  split_strategy: 26 matches
  train       : 23 matches
  validation  : 5 matches
  test        : 5 matches

Frames per split:


,data_split,frames,matches
0,test,292456,5
1,train,1397290,23
2,validation,296884,5


## 3. Create 5-frame non-overlapping windows

The tracking data runs at 10 Hz — one frame every 0.1 seconds.
Frames are grouped into **non-overlapping windows of exactly 5 frames** (0.5 seconds per window).

**Skip rule:**
- If a window has **fewer than 5 frames** (last partial window at the end of a period) → skip it.

Windows where ball coordinates are missing are kept. `ball_speed_avg_xy` will be `NaN` for those windows, which is valid — the model can learn from the absence of ball tracking data.

**`window_time`** counts elapsed time within each (match, period): window 1 ends at 0.5 s, window 2 at 1.0 s, and so on.
The counter resets at the start of every new (match, period).

In [170]:
# Illustrate window slicing on one (match, period)
WINDOW_SIZE = 5

sample_match  = str(df['t.match_id'].iloc[0])
sample_period = df.loc[df['t.match_id'] == sample_match, 't.period'].iloc[0]

period_df = (
    df[(df['t.match_id'] == sample_match) & (df['t.period'] == sample_period)]
    .sort_values('t.frame')
    .reset_index(drop=True)
)

n_frames    = len(period_df)
n_windows   = n_frames // WINDOW_SIZE
n_remainder = n_frames % WINDOW_SIZE

print(f'Match {sample_match}, period {sample_period}')
print(f'  Frames     : {n_frames}')
print(f'  Windows    : {n_windows}  (each = {WINDOW_SIZE} frames = 0.5 s)')
print(f'  Remainder  : {n_remainder} frame(s) discarded at end')
print()

for i in range(3):
    s = i * WINDOW_SIZE
    w = period_df.iloc[s : s + WINDOW_SIZE]
    window_time = (i + 1) * 0.5
    print(f'Window {i+1}  (window_time = {window_time} s)')
    display(
        w[['t.frame', 't.Videotimestamp', 'p.event_label',
           'p.dist_to_actual_event', 't.ball_x', 't.ball_y']]
        .reset_index(drop=True)
    )

Match 678949, period 1
  Frames     : 31658
  Windows    : 6331  (each = 5 frames = 0.5 s)
  Remainder  : 3 frame(s) discarded at end

Window 1  (window_time = 0.5 s)


,t.frame,t.Videotimestamp,p.event_label,p.dist_to_actual_event,t.ball_x,t.ball_y
0,0,1.0,PASS,0.2,NaN,NaN
1,1,1.1,PASS,0.1,NaN,NaN
2,2,1.2,PASS,0.0,NaN,NaN
3,3,1.3,PASS,0.1,NaN,NaN
4,4,1.4,PASS,0.2,NaN,NaN


Window 2  (window_time = 1.0 s)


,t.frame,t.Videotimestamp,p.event_label,p.dist_to_actual_event,t.ball_x,t.ball_y
0,5,1.5,PASS,0.3,NaN,NaN
1,6,1.6,PASS,0.4,NaN,NaN
2,7,1.7,PASS,0.5,NaN,NaN
3,8,1.8,PASS,0.6,NaN,NaN
4,9,1.9,PASS,0.7,NaN,NaN


Window 3  (window_time = 1.5 s)


,t.frame,t.Videotimestamp,p.event_label,p.dist_to_actual_event,t.ball_x,t.ball_y
0,10,2.0,PASS,0.8,NaN,NaN
1,11,2.1,PASS,0.9,NaN,NaN
2,12,2.2,no event,NaN,NaN,NaN
3,13,2.3,no event,NaN,NaN,NaN
4,14,2.4,no event,NaN,NaN,NaN


## 4. Event label selection

Each frame already carries a `p.event_label` from the pre-training step.
Within a window, multiple frames can be labeled with the same event (the pre-training ±1 s window is wider than 0.5 s).

**Rule:** Among frames where `p.event_label != "no event"`, pick the one with the **smallest `p.dist_to_actual_event`**.
This selects the frame temporally closest to the moment the event occurred.
Its label becomes the window's `p.event_label`. The `t.frame` of that row is stored as `primary_event_frame`.

If the window has no labeled frames at all, `p.event_label = "no event"` and `is_pass = 0`.

**Tie rule:** If two frames have exactly the same `p.dist_to_actual_event`, `pandas.idxmin()` picks the first occurrence — the labeled frame with the lower DataFrame index wins.

**Why `p.dist_to_actual_event`?** It is `|t.Videotimestamp − p.actual_event_frame|`, where `p.actual_event_frame` is the true event anchor from the events table. Choosing the minimum distance selects the frame that best represents the true event instant.

In [171]:
# Find the first window in the sample period that contains a labeled event
example_window = None
for i in range(n_windows):
    s = i * WINDOW_SIZE
    w = period_df.iloc[s : s + WINDOW_SIZE].reset_index(drop=True)
    if (w['p.event_label'] != 'no event').any():
        example_window = w
        example_idx = i
        break

if example_window is None:
    print('No labeled window found in this period.')
else:
    print(f'Window {example_idx + 1}  (window_time = {(example_idx + 1) * 0.5} s)')
    display(
        example_window[['t.frame', 't.Videotimestamp', 'p.event_label', 'p.dist_to_actual_event']]
    )

    events = example_window[example_window['p.event_label'] != 'no event']
    dist   = pd.to_numeric(events['p.dist_to_actual_event'], errors='coerce')
    closest_label = str(events.loc[dist.idxmin(), 'p.event_label'])
    primary_frame = int(events.loc[dist.idxmin(), 't.frame'])

    print(f'\np.event_label       = {closest_label}')
    print(f'is_pass             = {1 if closest_label == "PASS" else 0}')
    print(f'primary_event_frame = {primary_frame}')

Window 1  (window_time = 0.5 s)


,t.frame,t.Videotimestamp,p.event_label,p.dist_to_actual_event
0,0,1.0,PASS,0.2
1,1,1.1,PASS,0.1
2,2,1.2,PASS,0.0
3,3,1.3,PASS,0.1
4,4,1.4,PASS,0.2



p.event_label       = PASS
is_pass             = 1
primary_event_frame = 2


## 5. Ball speed (2D)

One feature is computed per window: **`ball_speed_avg_xy`**.

Within the 5-frame window, up to 4 frame-to-frame steps can be measured.
For each step where both frames have valid `(t.ball_x, t.ball_y)`, the 2D Euclidean distance is computed:

```
step_distance = sqrt((x[i+1] - x[i])^2 + (y[i+1] - y[i])^2)
```

`ball_speed_avg_xy` is the **mean** of all valid steps.
If no valid step exists (all ball positions are `NaN`), the value is `NaN`.

The unit is **metres per frame**. At 10 Hz, multiply by 10 to get m/s.

In [172]:
# Inline example on the first window of the sample period
window = period_df.iloc[:WINDOW_SIZE].reset_index(drop=True)

print('Ball positions (5 frames):')
display(window[['t.frame', 't.ball_x', 't.ball_y']].round(3))

speeds = []
for i in range(len(window) - 1):
    x1 = window.iloc[i]['t.ball_x']
    y1 = window.iloc[i]['t.ball_y']
    x2 = window.iloc[i + 1]['t.ball_x']
    y2 = window.iloc[i + 1]['t.ball_y']

    if pd.isna(x1) or pd.isna(y1) or pd.isna(x2) or pd.isna(y2):
        print(f'  step {i}->{i+1}: skipped (NaN)')
        continue

    d = float(np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2))
    speeds.append(d)
    print(f'  step {i}->{i+1}: {d:.4f} m/frame')

ball_speed_avg_xy = float(np.mean(speeds)) if speeds else float('nan')
print(f'\nball_speed_avg_xy = {ball_speed_avg_xy:.4f} m/frame  ({ball_speed_avg_xy * 10:.2f} m/s)')

Ball positions (5 frames):


,t.frame,t.ball_x,t.ball_y
0,0,NaN,NaN
1,1,NaN,NaN
2,2,NaN,NaN
3,3,NaN,NaN
4,4,NaN,NaN


  step 0->1: skipped (NaN)
  step 1->2: skipped (NaN)
  step 2->3: skipped (NaN)
  step 3->4: skipped (NaN)

ball_speed_avg_xy = nan m/frame  (nan m/s)


## 6. Closest player to the ball

For the **primary event frame** identified in section 4, the pipeline finds which visible player is closest to the ball at that exact moment.

**Steps:**
1. Take ball `(x, y)` from the primary event row — if either coordinate is missing, both outputs are `NaN`.
2. Loop over every player slot in the tracking data.
3. Skip any slot where `visible != True` or player coordinates are missing.
4. Compute the 2D Euclidean distance for each eligible player.
5. The slot with the smallest distance wins — its `player_id` and `team_id` become `closest_player_id` and `closest_player_team_id`.
6. If no eligible player exists, both outputs are `NaN`.

In [173]:
import re as _re

if example_window is None:
    print('No labeled window found — run section 4 first.')
else:
    # Identify the primary event row (minimum dist_to_actual_event)
    events = example_window[example_window['p.event_label'] != 'no event']
    dist_col = pd.to_numeric(events['p.dist_to_actual_event'], errors='coerce')
    primary_row = events.loc[dist_col.idxmin()]
    primary_frame = int(primary_row['t.frame'])

    ball_x = pd.to_numeric(primary_row['t.ball_x'], errors='coerce')
    ball_y = pd.to_numeric(primary_row['t.ball_y'], errors='coerce')

    print(f'Primary event frame : {primary_frame}')
    print(f'Event label         : {primary_row["p.event_label"]}')
    print(f'Ball position       : ({ball_x}, {ball_y})')

    if pd.isna(ball_x) or pd.isna(ball_y):
        print('\nBall position missing → closest_player_id = NaN, closest_player_team_id = NaN')
    else:
        # Discover player slots from column names
        player_x_cols = [c for c in df.columns if _re.match(r't\.player_\d+_x$', c)]
        slots = sorted(_re.search(r't\.player_(\d+)_x', c).group(1) for c in player_x_cols)

        candidates = []
        for slot in slots:
            vis_col = f't.player_{slot}_visible'
            if vis_col not in df.columns:
                continue
            vis_raw = primary_row[vis_col]
            visible = bool(vis_raw) if isinstance(vis_raw, bool) else str(vis_raw).lower() == 'true'
            if not visible:
                continue
            px = pd.to_numeric(primary_row[f't.player_{slot}_x'], errors='coerce')
            py = pd.to_numeric(primary_row[f't.player_{slot}_y'], errors='coerce')
            if pd.isna(px) or pd.isna(py):
                continue
            dist_m = float(np.sqrt((px - ball_x) ** 2 + (py - ball_y) ** 2))
            id_col   = f't.player_{slot}_id'
            team_col = f't.player_{slot}_team_id'
            candidates.append({
                'slot'      : slot,
                'player_id' : primary_row[id_col]   if id_col   in df.columns else None,
                'team'      : primary_row[team_col] if team_col in df.columns else None,
                'dist_m'    : round(dist_m, 3),
            })

        if not candidates:
            print('\nNo visible players → closest_player_id = NaN, closest_player_team_id = NaN')
        else:
            cdf = pd.DataFrame(candidates).sort_values('dist_m').reset_index(drop=True)
            print(f'\nVisible players : {len(cdf)}. Five closest:')
            display(cdf.head(5))
            winner = cdf.iloc[0]
            print(f'\nclosest_player_id   = {winner["player_id"]}')
            print(f'closest_player_team_id = {winner["team"]}')
            print(f'distance to ball    = {winner["dist_m"]} m')

Primary event frame : 2
Event label         : PASS
Ball position       : (nan, nan)

Ball position missing → closest_player_id = NaN, closest_player_team_id = NaN


## 7. Load the training tables

The full table is built by running `src/driblab/features/training_table.py` directly.
The script writes one parquet file per split. This notebook loads all three.

In [178]:
SPLIT_PATHS = {
    split: MODEL_BASE_DATA_DIR / f'training_table_{split}.parquet'
    for split in ['train', 'validation', 'test']
}

for split_name, path in SPLIT_PATHS.items():
    assert path.exists(), (
        f'Missing {path.name}.\n'
        'Run: python src/driblab/features/training_table.py'
    )

splits = {name: pd.read_parquet(path) for name, path in SPLIT_PATHS.items()}

print('Split sizes:')
for split_name, df in splits.items():
    n_pass = int(df['is_pass'].sum())
    print(f'  {split_name:12s}: {len(df):,} windows  |  PASS: {n_pass:,} ({n_pass/len(df)*100:.1f}%)')

print()
print('Sample from train:')
display(splits['train'].head(20))

Split sizes:
  train       : 279,437 windows  |  PASS: 88,363 (31.6%)
  validation  : 59,372 windows  |  PASS: 20,073 (33.8%)
  test        : 58,488 windows  |  PASS: 21,108 (36.1%)

Sample from train:


,t.match_id,t.period,window_time,primary_event_frame,data_split,p.event_label,is_pass,ball_speed_avg_xy,closest_player_id,closest_player_team_id
0,678949,1,0.5,2.0,train,PASS,1,NaN,NaN,NaN
1,678949,1,1.0,5.0,train,PASS,1,NaN,NaN,NaN
2,678949,1,1.5,10.0,train,PASS,1,NaN,NaN,NaN
3,678949,1,2.0,NaN,train,no event,0,NaN,NaN,NaN
4,678949,1,2.5,NaN,train,no event,0,NaN,NaN,NaN
5,678949,1,3.0,NaN,train,no event,0,1.277856,NaN,NaN
6,678949,1,3.5,NaN,train,no event,0,1.275727,NaN,NaN
7,678949,1,4.0,39.0,train,PASS,1,1.195799,1155966.0,1925
8,678949,1,4.5,44.0,train,PASS,1,0.758928,1166824.0,1925
9,678949,1,5.0,48.0,train,PASS,1,0.493233,1166824.0,1925


## 8. Validate

Check the output against expected criteria:
- Exactly 10 columns
- `ball_speed_avg_xy` may be `NaN` (ball untracked in window)
- `primary_event_frame`, `closest_player_id`, `closest_player_team_id` are `NaN` for `no event` windows
- `window_time` increments of 0.5

In [175]:
# Combine splits for a full-dataset validation view
all_windows = pd.concat(splits.values(), ignore_index=True)

print('=== VALIDATION ===\n')
print(f'Total windows : {len(all_windows):,}')
print(f'Columns       : {len(all_windows.columns)} (expected 10)')
print(f'  {list(all_windows.columns)}')

print('\nMissing values per column:')
missing = all_windows.isnull().sum()
for col, n in missing[missing > 0].items():
    print(f'  {col}: {n:,}')

print('\nData splits:')
for split_name in ['train', 'validation', 'test']:
    df_s = splits[split_name]
    n_pass = int(df_s['is_pass'].sum())
    print(f'  {split_name:12s}: {len(df_s):,} windows  |  PASS: {n_pass:,} ({n_pass/len(df_s)*100:.1f}%)')

print('\nWindow time (first 6 unique values):')
wt_sample = sorted(all_windows['window_time'].unique())[:6]
print(f'  {wt_sample}')

print('\nEvent types (train):')
for event, count in splits['train']['p.event_label'].value_counts().items():
    print(f'  {event}: {count:,}')

print('\nclosest_player_team_id — top values in train (event windows only):')
event_mask = splits['train']['p.event_label'] != 'no event'
print(splits['train'].loc[event_mask, 'closest_player_team_id'].value_counts().head(5).to_string())

=== VALIDATION ===

Total windows : 397,297
Columns       : 10 (expected 10)
  ['t.match_id', 't.period', 'window_time', 'primary_event_frame', 'data_split', 'p.event_label', 'is_pass', 'ball_speed_avg_xy', 'closest_player_id', 'closest_player_team_id']

Missing values per column:
  primary_event_frame: 230,124
  ball_speed_avg_xy: 172,051
  closest_player_id: 291,066
  closest_player_team_id: 291,066

Data splits:
  train       : 279,437 windows  |  PASS: 88,363 (31.6%)
  validation  : 59,372 windows  |  PASS: 20,073 (33.8%)
  test        : 58,488 windows  |  PASS: 21,108 (36.1%)

Window time (first 6 unique values):
  [np.float64(0.5), np.float64(1.0), np.float64(1.5), np.float64(2.0), np.float64(2.5), np.float64(3.0)]

Event types (train):
  no event: 164,677
  PASS: 88,363
  BALL TOUCH: 11,295
  TACKLE: 3,993
  AERIAL: 3,063
  BALL RECOVERY: 2,951
  TAKEON: 2,714
  FOUL: 2,381

closest_player_team_id — top values in train (event windows only):
closest_player_team_id
8561    5992
19

### Tracing a window back to the pre-training table

`primary_event_frame` stores the `t.frame` value of the exact row in `pre_training_table.parquet` that was selected as the primary event for that window.
To retrieve that row: filter the pre-training table on `(t.match_id, t.period, t.frame)`.

In [176]:
pre_training = pd.read_parquet(PRE_TRAINING_PATH)

# Pick the first PASS window in the train split as an example
example_row = splits['train'][splits['train']['p.event_label'] == 'PASS'].iloc[0]

match_id   = example_row['t.match_id']
period     = example_row['t.period']
frame_id   = example_row['primary_event_frame']
window_t   = example_row['window_time']

print(f'Training table row:  match={match_id}  period={period}  window_time={window_t}  primary_event_frame={frame_id}')
print()

# Look up the corresponding pre-training row
source_row = pre_training[
    (pre_training['t.match_id'] == match_id)
    & (pre_training['t.period'] == period)
    & (pre_training['t.frame'] == frame_id)
]

print('Pre-training row for this event:')
display(
    source_row[['t.match_id', 't.period', 't.frame', 't.Videotimestamp',
                'p.event_label', 'p.actual_event_frame', 'p.dist_to_actual_event']]
    .reset_index(drop=True)
)

Training table row:  match=678949  period=1  window_time=0.5  primary_event_frame=2.0

Pre-training row for this event:


,t.match_id,t.period,t.frame,t.Videotimestamp,p.event_label,p.actual_event_frame,p.dist_to_actual_event
0,678949,1,2,1.2,PASS,1.2,0.0


## 9. Output location

The file was already written by the script. No additional save step is needed from this notebook.

In [177]:
for split_name, path in SPLIT_PATHS.items():
    print(f'{split_name:12s}: {path.relative_to(PROJECT_ROOT)}')

train       : data/processed/model_base/training_table_train.parquet
validation  : data/processed/model_base/training_table_validation.parquet
test        : data/processed/model_base/training_table_test.parquet
